# Phase 6 (Abstractive Summarization) - Knowledge Distillation (student: Model 1 + LoRA)
Two trained runs act as **teachers** (default: the two Model 4 runs, LoRMA and LoRA). The **student** is the Phase-1 Model 1 with a fresh hand-written LoRA. On every training chunk the student learns from the gold summary (cross-entropy) and from the teachers' averaged next-token distribution (KL divergence, temperature `KD_TEMP`). Teachers are only run forward, never trained.

`loss = KD_ALPHA * CE(gold) + (1 - KD_ALPHA) * KD_TEMP^2 * KL(mean of teachers || student)`, computed on the summary positions only.

Inference is the same as in the other runs: summarize every chunk with the student, join in order. Chunk texts are saved while generating, so a time cap can be resumed.

**Attach (Kaggle "Add Input"):**
1. The Phase-6 data-prep output (`tokenized_summ.pkl`, `aligned_summ.pkl`, `summ_utils.py`)
2. Phase-1 Model 1 folder (`legal-gqa-final`) and `legal_gqa_model.py` (the student, and a teacher if you pick a Model 1 run)
3. Only if a Model 4 run is a teacher: Phase-1 Model 4 folder (`legal-gqamoe-final`) and `model.py`, `moe.py`, `legal_gqa_model.py`
4. The output folder of each of the two teacher runs: it must contain `kd_lora_weights.pt` or `lorma_weights.pt` (and `run_summary.json`, which gives the adapter settings)

Set `TEACHERS` in the config to the two runs you want (`m1_lora`, `m1_lorma`, `m4_lora`, `m4_lorma`).

**Output:** `/kaggle/working/summ_kd/` (student LoRA `kd_kd_lora_weights.pt`, predictions, `prediction.json`, `submission.zip`)

## 1. Configuration

In [ ]:
import os
import sys
import json
import glob
import math
import time
import random
import shutil
import pickle
import zipfile

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("SUM_DATA_DIR", None)
MODEL_DIR = os.environ.get("SUM_MODEL_DIR", None)
MODEL_PY = os.environ.get("SUM_MODEL_PY", None)
OUT_DIR = os.environ.get("SUM_OUT_DIR", "/kaggle/working/summ_kd")

MODEL_NAME_HINT = "legal-gqa"     # folder name fragment of the Phase-1 Model 1 folder
MODEL_PY_NAME = "legal_gqa_model.py"

SMOKE_TEST = False                # True = tiny run to check the code end to end
if os.environ.get("SUM_SMOKE") == "1":
    SMOKE_TEST = True
SMOKE_DOCS = 24
SMOKE_EVAL_DOCS = 3
SMOKE_TEST_DOCS = 3

SEED = 42
VAL_FOLD = 0                      # fold 0 is held out for validation (train on the other 4 folds)
EVAL_DOCS = 60                    # validation docs that get a full generated summary

# What to run
RUN_TRAIN = True
RUN_VAL = True
RUN_TEST = True
RESUME = True                     # pick up a checkpoint from an attached earlier run

# Knowledge distillation
TEACHERS = ["m4_lorma", "m4_lora"]   # one or two of: m1_lora, m1_lorma, m4_lora, m4_lorma
KD_ALPHA = 0.3                    # weight of the plain cross-entropy on the gold summary; the rest goes to the teachers
KD_TEMP = 2.0                     # softening temperature for student and teachers
TRAIN_BUDGET_HOURS = 6.0          # training stops after this long; the LR schedule is stretched to fit it

# Student LoRA (hand-written), on every linear layer inside the transformer blocks
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
LR = 2e-4
WEIGHT_DECAY = 0.0
EPOCHS = 2
WARMUP_FRAC = 0.03
MICRO_BATCH = 4                   # chunk examples per forward pass
GRAD_ACC = 4                      # effective batch = 16 examples
GRAD_CLIP = 1.0
EMPTY_KEEP_PROB = 0.5             # share of empty-target chunks kept in training (they teach "say nothing")
USE_CHECKPOINT = True             # gradient checkpointing, needed on a 16 GB T4
SAVE_EVERY_STEPS = 200
LOG_EVERY_STEPS = 20
SESSION_CAP_HOURS = 10.5          # the run saves and stops, and resumes on the next run

# Generation
GEN_BATCH = 8                     # chunks per generation batch (prefill builds full logits, keep it small)
LEN_MULT = 0.6                    # max new tokens per batch = LEN_MULT x the longest prompt (capped by the target budget)
REP_PENALTY = 1.0                 # 1.0 = off
ROUGE_L_MAX_WORDS = 1500          # ROUGE-L is computed on the first N words only (keeps the LCS fast)
USE_BERTSCORE = False             # needs the bert_score package and a downloaded scorer model; evaluation only

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
if SMOKE_TEST:
    EPOCHS = 1
    print("SMOKE_TEST is on: tiny data, 1 epoch, to check the code only.")
print("Config loaded.")

## 2. Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

START_TIME = time.time()
SESSION_CAP_SEC = SESSION_CAP_HOURS * 3600
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    if torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
        USE_SCALER = False
    else:
        AMP_DTYPE = torch.float16
        USE_SCALER = True
else:
    AMP_DTYPE = torch.float32
    USE_SCALER = False
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "| amp dtype:", AMP_DTYPE, "| grad scaler:", USE_SCALER)

## 3. Find the inputs

In [ ]:
def find_one_file(search_roots, file_name):
    for root in search_roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if file_name in file_names:
                return os.path.join(dir_path, file_name)
    return None


def find_model_dir(name_hint):
    for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
        low = dir_path.lower()
        if "config.json" in file_names and name_hint in low and "checkpoint" not in low and "moe" not in low:
            return dir_path
    return None


def find_m4_dir():
    for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
        low = dir_path.lower()
        if "config.json" not in file_names or "checkpoint" in low:
            continue
        model_type = ""
        try:
            model_type = json.load(open(os.path.join(dir_path, "config.json"))).get("model_type", "")
        except Exception:
            model_type = ""
        if model_type == "legal_gqa_mha_moe" or "gqamoe" in low:
            return dir_path
    return None


def find_m4_py():
    for root in ["/kaggle/input", "/kaggle/working"]:
        for dir_path, dir_names, file_names in os.walk(root):
            if "model.py" in file_names and "moe.py" in file_names and "legal_gqa_model.py" in file_names:
                return os.path.join(dir_path, "model.py")
    return None


if DATA_DIR is None:
    pkl_path = find_one_file(["/kaggle/input", "/kaggle/working"], "tokenized_summ.pkl")
    if pkl_path is None:
        raise FileNotFoundError("tokenized_summ.pkl not found. Attach the Phase-6 data-prep output.")
    DATA_DIR = os.path.dirname(pkl_path)

if MODEL_DIR is None:
    MODEL_DIR = find_model_dir(MODEL_NAME_HINT)
    if MODEL_DIR is None:
        raise FileNotFoundError("No Phase-1 Model 1 folder with config.json found. Attach legal-gqa-final.")

if MODEL_PY is None:
    MODEL_PY = find_one_file(["/kaggle/input", "/kaggle/working"], MODEL_PY_NAME)
    if MODEL_PY is None:
        raise FileNotFoundError("legal_gqa_model.py not found. Attach it (see the header).")

if len(TEACHERS) not in (1, 2):
    raise ValueError("TEACHERS must name one or two runs")
for t in TEACHERS:
    if t not in ("m1_lora", "m1_lorma", "m4_lora", "m4_lorma"):
        raise ValueError("unknown teacher " + t)
M4_DIR = None
M4_PY = None
if any(t.startswith("m4") for t in TEACHERS):
    M4_DIR = find_m4_dir()
    M4_PY = find_m4_py()
    if M4_DIR is None or M4_PY is None:
        raise FileNotFoundError("A Model 4 teacher needs the Phase-1 Model 4 folder (legal-gqamoe-final) and model.py + moe.py + legal_gqa_model.py in one folder.")

print("data dir:", DATA_DIR)
print("student (Model 1) dir:", MODEL_DIR)
print("student (Model 1) file:", MODEL_PY)
print("Model 4 dir / file (teachers):", M4_DIR, "/", M4_PY)

## 4. Load the data and build the chunk examples

In [ ]:
for name in ["summ_utils.py"]:
    if os.path.exists(os.path.join(DATA_DIR, name)):
        shutil.copy(os.path.join(DATA_DIR, name), name)
sys.path.insert(0, os.getcwd())
import summ_utils

with open(os.path.join(DATA_DIR, "tokenized_summ.pkl"), "rb") as f:
    DATA = pickle.load(f)
ALIGNED = summ_utils.load_aligned(os.path.join(DATA_DIR, "aligned_summ.pkl"))

META = DATA["meta"]
SEP_ID = int(META["sep_id"])
PAD_ID = int(META["pad_id"])
MODEL_CTX = int(META["model_ctx"])
CHUNK_DOC_LEN = int(META["chunk_doc_len"])
CHUNK_SUM_BUDGET = int(META["chunk_sum_budget"])
print("SEP:", SEP_ID, "| PAD:", PAD_ID, "| ctx:", MODEL_CTX, "| chunk doc:", CHUNK_DOC_LEN, "| target budget:", CHUNK_SUM_BUDGET)

# Pool = train + dev. Validation = the docs of VAL_FOLD.
TRAIN_ROWS = []
VAL_ROWS = []
for split in ["train", "dev"]:
    for row in DATA[split]:
        row["split"] = split
        if row["fold"] == VAL_FOLD:
            VAL_ROWS.append(row)
        else:
            TRAIN_ROWS.append(row)
TEST_ROWS = DATA["test"]

rng_docs = np.random.RandomState(SEED)
val_order = rng_docs.permutation(len(VAL_ROWS))
VAL_ROWS = [VAL_ROWS[i] for i in val_order]

if SMOKE_TEST:
    TRAIN_ROWS = TRAIN_ROWS[:SMOKE_DOCS]
    VAL_ROWS = VAL_ROWS[:SMOKE_EVAL_DOCS]
    TEST_ROWS = TEST_ROWS[:SMOKE_TEST_DOCS]
else:
    VAL_ROWS = VAL_ROWS[:EVAL_DOCS]

TRAIN_ALIGNED = [ALIGNED[row["split"]][row["id"]] for row in TRAIN_ROWS]

# One training example per (document, chunk). Empty targets are thinned out.
rng_pairs = np.random.RandomState(SEED)
PAIRS = []
PAIR_LEN = []
n_empty_seen = 0
n_empty_kept = 0
for ri in range(len(TRAIN_ROWS)):
    entry = TRAIN_ALIGNED[ri]
    bounds = entry["bounds"]
    for k in range(len(entry["targets"])):
        target_len = len(entry["targets"][k])
        if target_len == 0:
            n_empty_seen += 1
            if rng_pairs.rand() > EMPTY_KEEP_PROB:
                continue
            n_empty_kept += 1
        chunk_len = int(bounds[k + 1] - bounds[k])
        PAIRS.append((ri, k))
        PAIR_LEN.append(chunk_len + min(target_len, CHUNK_SUM_BUDGET))

print("train docs:", len(TRAIN_ROWS), "| val docs:", len(VAL_ROWS), "| test docs:", len(TEST_ROWS))
print("training examples:", len(PAIRS), "| empty targets kept:", n_empty_kept, "of", n_empty_seen)
print("example length: mean %.0f | max %d" % (np.mean(PAIR_LEN), max(PAIR_LEN)))


def make_batch(pair_list):
    seqs = []
    labs = []
    for ri, k in pair_list:
        row = TRAIN_ROWS[ri]
        entry = TRAIN_ALIGNED[ri]
        b = entry["bounds"]
        chunk = row["doc_ids"][b[k]:b[k + 1]]
        target = entry["targets"][k]
        inp, lab = summ_utils.build_chunk_example(chunk, target, SEP_ID, CHUNK_DOC_LEN, CHUNK_SUM_BUDGET)
        seqs.append(inp)
        labs.append(lab)
    T = max(len(s) for s in seqs)
    ids = np.full((len(seqs), T), PAD_ID, dtype=np.int64)
    labels = np.full((len(seqs), T), -100, dtype=np.int64)
    for i in range(len(seqs)):
        ids[i, :len(seqs[i])] = seqs[i]
        labels[i, :len(labs[i])] = labs[i]
    n_tok = int((labels[:, 1:] != -100).sum())
    return torch.from_numpy(ids), torch.from_numpy(labels), n_tok


def epoch_batches(epoch):
    rng = np.random.RandomState(SEED + epoch)
    order = rng.permutation(len(PAIRS))
    group = MICRO_BATCH * 50
    batches = []
    for g in range(0, len(order), group):
        idx = sorted(order[g:g + group].tolist(), key=lambda i: PAIR_LEN[i])
        for s in range(0, len(idx), MICRO_BATCH):
            batches.append([PAIRS[i] for i in idx[s:s + MICRO_BATCH]])
    rng.shuffle(batches)
    return batches


from transformers import AutoTokenizer
tok_dir = META["tokenizer_dir"]
if not os.path.exists(os.path.join(tok_dir, "tokenizer.json")):
    found = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    if len(found) == 0:
        raise FileNotFoundError("Tokenizer folder from the data-prep run not found. Attach the tokenizer dataset.")
    tok_dir = os.path.dirname(sorted(found)[0])
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
print("tokenizer:", tok_dir)

## 5. Load the frozen Phase-1 Model 1

In [ ]:
import importlib.util
import inspect

sys.path.insert(0, os.path.dirname(MODEL_PY))
spec = importlib.util.spec_from_file_location("legal_gqa_model", MODEL_PY)
legal_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(legal_mod)

LM, load_info = legal_mod.LegalGQAForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
missing_keys = []
for key in load_info.get("missing_keys", []):
    if key != "lm_head.weight":          # tied to the token embedding
        missing_keys.append(key)
unexpected_keys = list(load_info.get("unexpected_keys", []))
print("Phase-1 weights loaded | missing keys:", len(missing_keys), "| unexpected keys:", len(unexpected_keys))
if len(missing_keys) > 0:
    raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing_keys[:5]))
if LM.config.vocab_size != META["vocab_size"]:
    print("WARNING: model vocab", LM.config.vocab_size, "!= tokenizer vocab", META["vocab_size"])
if LM.config.vocab_size <= SEP_ID:
    raise RuntimeError("tokenizer / model vocab mismatch")

# Phase-1 saves only the token embedding when the output layer is tied to it. If lm_head.weight was missing from the
# checkpoint it must be tied explicitly, otherwise the output layer stays randomly initialised.
if "lm_head.weight" in list(load_info.get("missing_keys", [])):
    LM.lm_head.weight = LM.wte.weight
    print("lm_head.weight was not in the checkpoint -> tied to the token embedding")
tie_gap = float((LM.lm_head.weight - LM.wte.weight).abs().max())
print("lm_head vs token embedding max abs gap: %.6f (0 means tied)" % tie_gap)

LM.float()
LM = LM.to(DEVICE)
LM.eval()
for p in LM.parameters():
    p.requires_grad = False

N_BASE = sum(p.numel() for p in LM.parameters())
print("model:", type(LM).__name__, "| params: %.2fM" % (N_BASE / 1e6))
print("forward arguments:", list(inspect.signature(LM.forward).parameters.keys()))


def backbone_hidden(input_ids):
    # Same computation as the Phase-1 forward(), stopped before lm_head. Causal model, right padding is harmless.
    T = input_ids.shape[1]
    x = LM.drop(LM.wte(input_ids))
    cos, sin = LM._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in LM.blocks:
        if torch.is_grad_enabled() and USE_CHECKPOINT:
            def run(h, blk=blk):
                return blk(h, cos, sin, None, True, None, False)[0]
            x = torch_checkpoint(run, x, use_reentrant=False)
        else:
            x = blk(x, cos, sin, None, True, None, False)[0]
    return LM.ln_f(x)


# My manual forward must agree with the model's own forward.
with torch.no_grad():
    probe = torch.tensor(np.random.randint(3, LM.config.vocab_size, size=(1, 48)), device=DEVICE)
    own_logits = LM.lm_head(backbone_hidden(probe)).float()
    ref_logits = LM(input_ids=probe).logits.float()
    agree = float((own_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean())
    diff = float((own_logits - ref_logits).abs().max())
print("manual forward vs model forward: top-1 agreement %.3f | max logit diff %.4f" % (agree, diff))
assert agree > 0.9, "manual forward does not match the model forward"

## 6. Hand-written LoRA

LoRA: `y = W0 x + (alpha / r) * B A x`. `W0` is frozen, `B` starts at zero, so training starts exactly at the Phase-1 weights.

In [ ]:
def hand_dropout(x, p, training):
    # dropout written by hand: zero some entries, scale the rest
    if (not training) or p == 0.0:
        return x
    keep_prob = 1.0 - p
    rand = torch.rand_like(x)
    mask = (rand < keep_prob).to(x.dtype)
    return x * mask / keep_prob


def hand_uniform_(tensor, bound):
    # fill a tensor with numbers drawn uniformly from [-bound, bound]
    with torch.no_grad():
        tensor.copy_(torch.rand_like(tensor) * 2.0 * bound - bound)


class LoRALinear(nn.Module):
    # y = W0 x + (alpha / r) * B (A x)       W0 frozen, only A and B are trained
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.d_in = d_in
        self.r = r
        self.scale = alpha / r
        self.p_drop = dropout
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, d_in, device=dev, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, device=dev, dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self):
        # A: uniform in [-1/sqrt(d_in), 1/sqrt(d_in)], B: zeros, so the update starts at exactly 0
        bound = 1.0 / math.sqrt(self.d_in)
        hand_uniform_(self.lora_A, bound)
        with torch.no_grad():
            self.lora_B.zero_()

    def forward(self, x):
        y = self.base(x)
        x32 = x.float()
        x32 = hand_dropout(x32, self.p_drop, self.training)
        low = x32 @ self.lora_A.t()        # (.., r)
        delta = low @ self.lora_B.t()      # (.., d_out)
        delta = delta * self.scale
        return y + delta.to(y.dtype)


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


# Targets: every linear layer inside the blocks. Not the embedding, not lm_head.
LORA_TARGETS = []
for name, module in LM.named_modules():
    if isinstance(module, nn.Linear) and name.startswith("blocks."):
        LORA_TARGETS.append(name)
if len(LORA_TARGETS) == 0:
    raise RuntimeError("No linear layers found under blocks. Check the Model 1 module names.")
print("LoRA target layers:", len(LORA_TARGETS), "| first few:", LORA_TARGETS[:5])

with torch.no_grad():
    check_ids = torch.tensor([[5, 9, 12, 40, 77, SEP_ID]], device=DEVICE)
    before = backbone_hidden(check_ids).float().clone()

LORA_MODULES = []
for name in LORA_TARGETS:
    wrapped = LoRALinear(LM.get_submodule(name), LORA_R, LORA_ALPHA, LORA_DROPOUT)
    set_submodule(LM, name, wrapped)
    LORA_MODULES.append(wrapped)

with torch.no_grad():
    after = backbone_hidden(check_ids).float()
identity_diff = float((before - after).abs().max())
print("identity check (max abs diff at init, should be 0):", identity_diff)
assert identity_diff < 1e-4, "LoRA changed the Phase-1 output at init"

LORA_PARAMS = []
for m in LORA_MODULES:
    LORA_PARAMS.append(m.lora_A)
    LORA_PARAMS.append(m.lora_B)
N_LORA = sum(p.numel() for p in LORA_PARAMS)
print("trainable LoRA params: %s (%.2f%% of the model)" % (f"{N_LORA:,}", 100.0 * N_LORA / N_BASE))


def set_train_mode():
    LM.eval()                      # Phase-1 dropout stays off
    for m in LORA_MODULES:
        m.train()                  # LoRA dropout on


def set_eval_mode():
    LM.eval()
    for m in LORA_MODULES:
        m.eval()


def lora_state():
    return {name: (m.lora_A.detach().cpu(), m.lora_B.detach().cpu()) for name, m in zip(LORA_TARGETS, LORA_MODULES)}


def load_lora_state(state):
    for name, m in zip(LORA_TARGETS, LORA_MODULES):
        a, b = state[name]
        with torch.no_grad():
            m.lora_A.copy_(a.to(m.lora_A.device))
            m.lora_B.copy_(b.to(m.lora_B.device))

## 6b. Teachers

Each teacher is a frozen Phase-1 model with its trained adapter weights put back in (found by looking at the attached output folders). They are only used to produce next-token distributions.

In [ ]:
_ROT_CACHE = {}
_EYE_CACHE = {}


def hand_dropout(x, p, training):
    # dropout written by hand: zero some entries, scale the rest
    if (not training) or p == 0.0:
        return x
    keep_prob = 1.0 - p
    rand = torch.rand_like(x)
    mask = (rand < keep_prob).to(x.dtype)
    return x * mask / keep_prob


def hand_uniform_(tensor, bound):
    # fill a tensor with numbers drawn uniformly from [-bound, bound]
    with torch.no_grad():
        tensor.copy_(torch.rand_like(tensor) * 2.0 * bound - bound)


def get_rot_idx(d, device):
    # index table for the row rotation: entry (i, j) = (j - i) mod d
    # shared by all layers with the same output size
    key = (d, str(device))
    if key not in _ROT_CACHE:
        rows = torch.arange(d, device=device).unsqueeze(1)
        cols = torch.arange(d, device=device).unsqueeze(0)
        diff = cols - rows
        _ROT_CACHE[key] = diff % d
    return _ROT_CACHE[key]


def get_eye(d, device):
    # identity matrix built by hand: zeros with ones on the diagonal
    key = (d, str(device))
    if key not in _EYE_CACHE:
        eye = torch.zeros(d, d, device=device)
        diag = torch.arange(d, device=device)
        eye[diag, diag] = 1.0
        _EYE_CACHE[key] = eye
    return _EYE_CACHE[key]


def rotate_rows(M):
    # row i of M is shifted right by i places (wrapping around). Written with plain index tensors.
    d = M.shape[0]
    rows = torch.arange(d, device=M.device).unsqueeze(1).expand(d, d)
    cols = get_rot_idx(d, M.device)
    return M[rows, cols]




class FrozenLoRA(nn.Module):
    # inference-only copy of the hand-written LoRA layer: y = W0 x + scale * B (A x)
    def __init__(self, base, A, B, scale):
        super().__init__()
        self.base = base
        self.register_buffer("A", A.float())
        self.register_buffer("B", B.float())
        self.scale = scale
        self.enabled = True

    def forward(self, x):
        y = self.base(x)
        if not self.enabled:
            return y
        delta = (x.float() @ self.A.t()) @ self.B.t()
        return y + (delta * self.scale).to(y.dtype)


class FrozenLoRMA(nn.Module):
    # inference-only copy of the hand-written LoRMA layer (plus or perm)
    def __init__(self, base, A, B, scale, variant):
        super().__init__()
        self.base = base
        self.register_buffer("A", A.float())
        self.register_buffer("B", B.float())
        self.scale = scale
        self.variant = variant
        self.d = base.weight.shape[0]
        self.enabled = True

    def forward(self, x):
        z = self.base(x)
        if not self.enabled:
            return z
        if self.variant == "plus":
            delta = (z.float() @ self.A.t()) @ self.B.t()
            return z + (delta * self.scale).to(z.dtype)
        with torch.autocast(device_type=z.device.type, enabled=False):
            M = self.B @ self.A
            R = rotate_rows(M)
            D = R - get_eye(self.d, M.device)
        return z + z @ D.t()


def find_teacher_runs():
    # Looks at every attached adapter file and works out which run it belongs to.
    # Final weights (lora_weights.pt / lorma_weights.pt) win; an unfinished run's checkpoint (ckpt_*.pt) is the fallback.
    found = {}
    for pass_no in (0, 1):
        for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
            if pass_no == 0:
                names = (("lora_weights.pt", "lora"), ("lorma_weights.pt", "lorma"))
            else:
                names = (("ckpt_lora.pt", "lora"), ("ckpt_lorma.pt", "lorma"))
            for fname, kind in names:
                if fname not in file_names:
                    continue
                path = os.path.join(dir_path, fname)
                summ = {}
                sp = os.path.join(dir_path, "run_summary.json")
                if os.path.exists(sp):
                    try:
                        summ = json.load(open(sp))
                    except Exception:
                        summ = {}
                state = torch.load(path, map_location="cpu", weights_only=False)
                if pass_no == 1:
                    state = state[kind]
                arch = None
                model_text = str(summ.get("model", ""))
                if "Model 4" in model_text:
                    arch = "m4"
                elif "Model 1" in model_text:
                    arch = "m1"
                if arch is None:
                    deep = [k for k in state.keys() if len(k.split(".")) > 4]
                    arch = "m4" if len(deep) > 0 else "m1"
                key = arch + "_" + kind
                if key not in found:
                    found[key] = {"path": path, "summary": summ, "from_ckpt": pass_no == 1}
    return found


TEACHER_RUNS = find_teacher_runs()
print("adapter runs found in the inputs:", {k: v["path"] for k, v in TEACHER_RUNS.items()})
for t in TEACHERS:
    if t not in TEACHER_RUNS:
        raise FileNotFoundError("Teacher %s: no matching weights found (looked for %s or ckpt_%s.pt). Attach that run's output folder, "
                                "or change TEACHERS to the runs you have: %s" % (
            t, "lora_weights.pt" if t.endswith("_lora") else "lorma_weights.pt", t.split("_")[1], list(TEACHER_RUNS.keys())))


def load_backbone(arch):
    if arch == "m1":
        model, info = legal_mod.LegalGQAForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
    else:
        sys.path.insert(0, os.path.dirname(M4_PY))
        spec4 = importlib.util.spec_from_file_location("legal_moe_model", M4_PY)
        mod4 = importlib.util.module_from_spec(spec4)
        sys.modules["legal_moe_model"] = mod4
        spec4.loader.exec_module(mod4)
        model, info = mod4.LegalGQAMoEForCausalLM.from_pretrained(M4_DIR, output_loading_info=True)
    miss = [k for k in info.get("missing_keys", []) if k != "lm_head.weight"]
    if len(miss) > 0:
        raise RuntimeError("teacher backbone not fully loaded, missing: " + str(miss[:5]))
    if "lm_head.weight" in list(info.get("missing_keys", [])):
        model.lm_head.weight = model.wte.weight
    model.float()
    model = model.to(DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model


TEACHER_MODELS = []
TEACHER_ADAPTERS = []
for t in TEACHERS:
    arch, kind = t.split("_")
    run = TEACHER_RUNS[t]
    state = torch.load(run["path"], map_location="cpu", weights_only=False)
    if run["from_ckpt"]:
        state = state[kind]
        print("NOTE: teacher %s comes from a training checkpoint, not from finished weights" % t)
    cfg = run["summary"].get(kind, {})
    variant = cfg.get("variant", "perm")
    alpha = float(cfg.get("alpha", 32 if kind == "lora" else 16))
    tm = load_backbone(arch)
    adapters = []
    for name in state.keys():
        a, b = state[name]
        base = tm.get_submodule(name)
        scale = alpha / a.shape[0]
        if kind == "lora":
            wrapped = FrozenLoRA(base, a.to(DEVICE), b.to(DEVICE), scale)
        else:
            wrapped = FrozenLoRMA(base, a.to(DEVICE), b.to(DEVICE), scale, variant)
        set_submodule(tm, name, wrapped)
        adapters.append(wrapped)
    TEACHER_MODELS.append(tm)
    TEACHER_ADAPTERS.append(adapters)
    print("teacher %s: %s | %d adapted layers | r=%d alpha=%.0f%s | weights %s" % (
        t, type(tm).__name__, len(adapters), a.shape[0], alpha, (" variant=" + variant) if kind == "lorma" else "", run["path"]))
    del state


@torch.no_grad()
def hidden_generic(model, input_ids):
    # same computation as the Phase-1 forward(), stopped before lm_head (causal, right padding is harmless)
    T = input_ids.shape[1]
    x = model.drop(model.wte(input_ids))
    cos, sin = model._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in model.blocks:
        x = blk(x, cos, sin, None, True, None, False)[0]
    return model.ln_f(x)


@torch.no_grad()
def teacher_selected_logits(tm, input_ids, selected):
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        h = hidden_generic(tm, input_ids)
        h_sel = h[:, :-1][selected]
        logits = tm.lm_head(h_sel)
    return logits.float()

## 7. Training

In [ ]:
KD_LOG = {"ce": 0.0, "kd": 0.0, "tok": 0}


def lm_loss_sum(input_ids, labels):
    # Sum over the supervised (summary) positions of: KD_ALPHA * CE(gold) + (1 - KD_ALPHA) * T^2 * KL(teachers || student)
    input_ids = input_ids.to(DEVICE)
    labels = labels.to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        hidden = backbone_hidden(input_ids)
    shifted_hidden = hidden[:, :-1]
    target = labels[:, 1:]
    selected = target != -100
    h_sel = shifted_hidden[selected]
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        s_logits = LM.lm_head(h_sel)
    s_logits = s_logits.float()
    gold = target[selected]
    ce = F.cross_entropy(s_logits, gold, reduction="sum")
    if KD_ALPHA >= 1.0 or len(TEACHER_MODELS) == 0:
        return ce

    t_prob = None
    for tm in TEACHER_MODELS:
        t_logits = teacher_selected_logits(tm, input_ids, selected)
        p = F.softmax(t_logits / KD_TEMP, dim=-1)
        t_prob = p if t_prob is None else t_prob + p
        del t_logits
    t_prob = t_prob / len(TEACHER_MODELS)
    log_s = F.log_softmax(s_logits / KD_TEMP, dim=-1)
    kd = F.kl_div(log_s, t_prob, reduction="sum") * (KD_TEMP ** 2)
    KD_LOG["ce"] += float(ce.detach())
    KD_LOG["kd"] += float(kd.detach())
    KD_LOG["tok"] += int(gold.numel())
    return KD_ALPHA * ce + (1.0 - KD_ALPHA) * kd


# Sanity check: the teachers must be better than the plain Phase-1 model on the summary tokens, and the student starts at the plain model.
with torch.no_grad():
    chk = epoch_batches(0)[:6]
    sums = {"student": 0.0}
    for i in range(len(TEACHER_MODELS)):
        sums["T%d on" % i] = 0.0
        sums["T%d off" % i] = 0.0
    n_tok_chk = 0
    for group in chk:
        ids, labels, n_tok = make_batch(group)
        ids = ids.to(DEVICE)
        labels = labels.to(DEVICE)
        target = labels[:, 1:]
        selected = target != -100
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            h_sel = backbone_hidden(ids)[:, :-1][selected]
            sl = LM.lm_head(h_sel)
        sums["student"] += float(F.cross_entropy(sl.float(), target[selected], reduction="sum"))
        for i, tm in enumerate(TEACHER_MODELS):
            for flag, tag in ((True, "on"), (False, "off")):
                for ad in TEACHER_ADAPTERS[i]:
                    ad.enabled = flag
                tl = teacher_selected_logits(tm, ids, selected)
                sums["T%d %s" % (i, tag)] += float(F.cross_entropy(tl, target[selected], reduction="sum"))
            for ad in TEACHER_ADAPTERS[i]:
                ad.enabled = True
        n_tok_chk += int(selected.sum())
print("cross-entropy per summary token on %d training batches:" % len(chk))
for key in sums:
    print("  %-10s %.4f" % (key, sums[key] / max(n_tok_chk, 1)))
for i, t in enumerate(TEACHERS):
    if sums["T%d on" % i] >= sums["T%d off" % i]:
        print("WARNING: teacher %s is not better with its adapter than without it. Check that the right weights were attached." % t)


CKPT_PATH = os.path.join(OUT_DIR, "ckpt_kd.pt")
LORA_PATH = os.path.join(OUT_DIR, "kd_lora_weights.pt")
STOPPED_EARLY = False
BUDGET_REACHED = False
TRAIN_SECONDS = 0.0
LAST_LOSS = None

if RUN_TRAIN and len(PAIRS) > 0:
    micro_per_epoch = int(math.ceil(len(PAIRS) / MICRO_BATCH))
    steps_per_epoch = int(math.ceil(micro_per_epoch / GRAD_ACC))
    total_steps = EPOCHS * steps_per_epoch
    warm_steps = max(1, int(WARMUP_FRAC * total_steps))
    print("micro-batches per epoch:", micro_per_epoch, "| optimizer steps per epoch:", steps_per_epoch, "| total steps:", total_steps)

    optimizer = torch.optim.AdamW(LORA_PARAMS, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    def lr_at(step):
        if step < warm_steps:
            return LR * (step + 1) / warm_steps
        progress = (step - warm_steps) / max(1, total_steps - warm_steps)
        time_progress = (seconds_before + (time.time() - t_run)) / (TRAIN_BUDGET_HOURS * 3600.0)
        progress = min(1.0, max(progress, time_progress))
        return LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

    state = {"epoch": 0, "micro": 0, "step": 0, "seconds": 0.0}

    if RESUME:
        found = None
        if os.path.exists(CKPT_PATH):
            found = CKPT_PATH
        else:
            found = find_one_file(["/kaggle/input"], "ckpt_kd.pt")
        if found is not None:
            ck = torch.load(found, map_location="cpu", weights_only=False)
            load_lora_state(ck["lora"])
            optimizer.load_state_dict(ck["opt"])
            if USE_SCALER and ck.get("scaler") is not None:
                scaler.load_state_dict(ck["scaler"])
            state = ck["state"]
            print("resumed from", found, "| epoch", state["epoch"], "| micro", state["micro"], "| step", state["step"])
            if found != CKPT_PATH:
                shutil.copy(found, CKPT_PATH)

    def save_ckpt():
        torch.save({"lora": lora_state(), "opt": optimizer.state_dict(),
                    "scaler": scaler.state_dict() if USE_SCALER else None, "state": state}, CKPT_PATH)

    set_train_mode()
    run_loss = 0.0
    run_tok = 0
    t_run = time.time()
    t_last_log = time.time()
    seconds_before = state["seconds"]

    for epoch in range(state["epoch"], EPOCHS):
        batches = epoch_batches(epoch)
        start_micro = state["micro"] if epoch == state["epoch"] else 0
        micro = start_micro
        while micro < len(batches):
            group = batches[micro:micro + GRAD_ACC]
            built = [make_batch(b) for b in group]
            total_tokens = max(1, sum(x[2] for x in built))

            for g in optimizer.param_groups:
                g["lr"] = lr_at(state["step"])

            group_loss = 0.0
            for ids, labels, n_tok in built:
                loss_sum = lm_loss_sum(ids, labels)
                group_loss += float(loss_sum.detach())
                scaler.scale(loss_sum / total_tokens).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(LORA_PARAMS, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            micro += len(group)
            state["epoch"] = epoch
            state["micro"] = micro
            state["step"] += 1
            run_loss += group_loss
            run_tok += total_tokens
            LAST_LOSS = group_loss / total_tokens

            if state["step"] % LOG_EVERY_STEPS == 0:
                elapsed = time.time() - t_last_log
                per_step = elapsed / LOG_EVERY_STEPS
                left = total_steps - state["step"]
                print("epoch %d | step %d/%d | loss %.4f | ce %.3f kd %.3f | lr %.2e | %.1f s/step | eta %.0f min" % (
                    epoch, state["step"], total_steps, run_loss / max(run_tok, 1),
                    KD_LOG["ce"] / max(KD_LOG["tok"], 1), KD_LOG["kd"] / max(KD_LOG["tok"], 1), lr_at(state["step"]),
                    per_step, left * per_step / 60.0))
                KD_LOG["ce"] = 0.0
                KD_LOG["kd"] = 0.0
                KD_LOG["tok"] = 0
                run_loss = 0.0
                run_tok = 0
                t_last_log = time.time()

            state["seconds"] = seconds_before + (time.time() - t_run)
            if state["step"] % SAVE_EVERY_STEPS == 0:
                save_ckpt()
            if (seconds_before + (time.time() - t_run)) > TRAIN_BUDGET_HOURS * 3600.0:
                BUDGET_REACHED = True
                print("Training time budget (%.1f h) reached at step %d; stopping with the schedule fully decayed." % (TRAIN_BUDGET_HOURS, state["step"]))
                break
            if time.time() - START_TIME > SESSION_CAP_SEC:
                save_ckpt()
                STOPPED_EARLY = True
                print("Session time cap reached. Checkpoint saved. Attach this notebook's output as an input and run again to resume.")
                break
        if STOPPED_EARLY or BUDGET_REACHED:
            break
        state["epoch"] = epoch + 1
        state["micro"] = 0
        save_ckpt()
        print("epoch", epoch, "finished")

    TRAIN_SECONDS = state["seconds"]
    if not STOPPED_EARLY:
        state["epoch"] = EPOCHS
        save_ckpt()
        torch.save(lora_state(), LORA_PATH)
        print("training done | %.1f min | last loss %.4f" % (TRAIN_SECONDS / 60.0, LAST_LOSS if LAST_LOSS is not None else float("nan")))
else:
    if os.path.exists(LORA_PATH):
        load_lora_state(torch.load(LORA_PATH, map_location="cpu"))
        print("loaded LoRA weights from", LORA_PATH)
    else:
        found = find_one_file(["/kaggle/input"], "kd_lora_weights.pt")
        if found is not None:
            load_lora_state(torch.load(found, map_location="cpu"))
            print("loaded LoRA weights from", found)
        else:
            print("No training and no saved LoRA weights: the model is the plain Phase-1 model.")

## 8. Generation (chunk by chunk, KV cache, greedy)

Own decoding loop over the model's forward (left padding for batches). First a check that padding and the cache do not change the logits.

In [ ]:
@torch.no_grad()
def generate_batch(prompts, max_new):
    # prompts: list of 1-D int arrays, each ending with SEP. Greedy decoding until SEP. Returns a list of id lists.
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((B, L), dtype=torch.long)
    for i, p in enumerate(prompts):
        ids[i, L - len(p):] = torch.as_tensor(np.asarray(p, dtype=np.int64))
        mask[i, L - len(p):] = 1
    ids = ids.to(DEVICE)
    mask = mask.to(DEVICE)

    out = LM(input_ids=ids, attention_mask=mask, use_cache=True)
    past = out.past_key_values
    logits = out.logits[:, -1].float()
    del out

    generated = torch.zeros((B, max_new), dtype=torch.long, device=DEVICE)
    finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    n_steps = 0
    for step in range(max_new):
        if REP_PENALTY != 1.0 and step > 0:
            prev = generated[:, :step]
            score = logits.gather(1, prev)
            score = torch.where(score < 0, score * REP_PENALTY, score / REP_PENALTY)
            logits = logits.scatter(1, prev, score)
        nxt = logits.argmax(dim=-1)
        nxt = torch.where(finished, torch.full_like(nxt, SEP_ID), nxt)
        generated[:, step] = nxt
        finished = finished | (nxt == SEP_ID)
        n_steps = step + 1
        if bool(finished.all()):
            break
        mask = torch.cat([mask, torch.ones((B, 1), dtype=torch.long, device=DEVICE)], dim=1)
        out = LM(input_ids=nxt.unsqueeze(1), attention_mask=mask, past_key_values=past, use_cache=True)
        past = out.past_key_values
        logits = out.logits[:, -1].float()
        del out

    results = []
    gen_cpu = generated[:, :n_steps].cpu().numpy()
    for i in range(B):
        seq = gen_cpu[i].tolist()
        if SEP_ID in seq:
            seq = seq[:seq.index(SEP_ID)]
        results.append(seq)
    return results


set_eval_mode()
LM.config.use_cache = True

# Check 1: left padding must not change the next-token logits. Check 2: the KV cache must match a full recompute.
probe_row = TRAIN_ROWS[0]
probe_ids = np.asarray(probe_row["doc_ids"][:120]).astype(np.int64)
probe_ids = np.concatenate([probe_ids, [SEP_ID]])
with torch.no_grad():
    single = torch.as_tensor(probe_ids, device=DEVICE).unsqueeze(0)
    ref = LM(input_ids=single).logits[0, -1].float()

    pad_n = 9
    padded = torch.cat([torch.full((1, pad_n), PAD_ID, dtype=torch.long, device=DEVICE), single], dim=1)
    pmask = torch.cat([torch.zeros((1, pad_n), dtype=torch.long, device=DEVICE), torch.ones_like(single)], dim=1)
    pad_logits = LM(input_ids=padded, attention_mask=pmask).logits[0, -1].float()
    pad_diff = float((ref - pad_logits).abs().max())

    o1 = LM(input_ids=single, use_cache=True)
    nxt_tok = o1.logits[:, -1].argmax(-1, keepdim=True)
    o2 = LM(input_ids=nxt_tok, past_key_values=o1.past_key_values, use_cache=True)
    full_next = LM(input_ids=torch.cat([single, nxt_tok], dim=1)).logits[0, -1].float()
    cache_diff = float((o2.logits[0, -1].float() - full_next).abs().max())

print("left-padding max logit diff: %.4f | KV-cache max logit diff: %.4f" % (pad_diff, cache_diff))
print("(small numbers are fine; fp16 gives differences of a few hundredths)")
assert pad_diff < 2.0, "left padding changes the logits a lot: the attention-mask path is not behaving"
assert cache_diff < 2.0, "KV cache disagrees with a full recompute: check how the model returns past_key_values"

t_probe = time.time()
test_out = generate_batch([probe_ids, probe_ids[:60].copy()], 8)
print("generation smoke:", [len(x) for x in test_out], "tokens in %.2f s" % (time.time() - t_probe))

In [ ]:
GEN_STATS = {"tokens": 0, "seconds": 0.0}
GEN_INCOMPLETE = False

# Fingerprint of the adapter weights, so a cached chunk text is only reused for the very same adapter.
ADAPTER_FINGERPRINT = "%.6f|%d" % (sum(float(p.detach().abs().sum()) for p in LORA_PARAMS), len(LORA_PARAMS))
print("adapter fingerprint:", ADAPTER_FINGERPRINT)


def load_chunk_cache(label):
    # Looks in this notebook's output first, then in attached earlier outputs. Only a matching fingerprint is accepted.
    name = "chunks_%s.json" % label
    paths = [os.path.join(OUT_DIR, name)]
    for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
        if name in file_names:
            paths.append(os.path.join(dir_path, name))
    texts = {}
    for path in paths:
        if not os.path.exists(path):
            continue
        try:
            blob = json.load(open(path, encoding="utf-8"))
        except Exception:
            continue
        if blob.get("fingerprint") == ADAPTER_FINGERPRINT and blob.get("smoke") == bool(SMOKE_TEST):
            for key in blob["texts"]:
                texts[key] = blob["texts"][key]
            print("  cache:", len(blob["texts"]), "chunk texts from", path)
    return texts


def save_chunk_cache(label, texts):
    path = os.path.join(OUT_DIR, "chunks_%s.json" % label)
    blob = {"fingerprint": ADAPTER_FINGERPRINT, "smoke": bool(SMOKE_TEST), "texts": texts}
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(blob, f, ensure_ascii=False)
    os.replace(path + ".tmp", path)


def summarize_documents(rows, label):
    # Every chunk of every document is summarized; pieces are joined in order.
    # Finished chunk texts are saved (chunks_<label>.json) so an interrupted run can continue.
    # Returns None when the session time cap is reached before everything is generated.
    global GEN_INCOMPLETE
    cache = load_chunk_cache(label)
    jobs = []
    n_chunks_all = 0
    for d, row in enumerate(rows):
        entry = ALIGNED[row["split"]][row["id"]] if "split" in row else ALIGNED["test"][row["id"]]
        pieces = summ_utils.chunk_slices(row["doc_ids"], entry["bounds"])
        for k, piece in enumerate(pieces):
            n_chunks_all += 1
            key = "%s|%d" % (row["id"], k)
            if key in cache:
                continue
            prompt = summ_utils.build_chunk_prompt(piece, SEP_ID, CHUNK_DOC_LEN)
            jobs.append((key, prompt))
    print(label, "| documents:", len(rows), "| chunks in total:", n_chunks_all, "| still to generate:", len(jobs))
    jobs.sort(key=lambda j: -len(j[1]))

    t0 = time.time()
    total_new = 0
    n_batch = 0
    for start in range(0, len(jobs), GEN_BATCH):
        if time.time() - START_TIME > SESSION_CAP_SEC:
            save_chunk_cache(label, cache)
            GEN_INCOMPLETE = True
            print("Session time cap reached. Chunk texts saved. Attach this notebook's output as an input and run again to continue.")
            return None
        batch = jobs[start:start + GEN_BATCH]
        longest = max(len(j[1]) for j in batch)
        max_new = int(min(CHUNK_SUM_BUDGET, max(16, LEN_MULT * longest)))
        outs = generate_batch([j[1] for j in batch], max_new)
        for (key, prompt), seq in zip(batch, outs):
            cache[key] = tokenizer.decode(seq, clean_up_tokenization_spaces=False) if len(seq) > 0 else ""
            total_new += len(seq)
        n_batch += 1
        if n_batch % 20 == 0:
            save_chunk_cache(label, cache)
        if (start // GEN_BATCH) % 25 == 0:
            print("  generated", min(start + GEN_BATCH, len(jobs)), "/", len(jobs), "| %.0f s" % (time.time() - t0))
    save_chunk_cache(label, cache)
    seconds = time.time() - t0
    GEN_STATS["tokens"] += total_new
    GEN_STATS["seconds"] += seconds
    if len(jobs) > 0:
        print(label, "done | %d new tokens | %.1f s | %.1f tokens/s" % (total_new, seconds, total_new / max(seconds, 1e-6)))

    summaries = []
    for d, row in enumerate(rows):
        entry = ALIGNED[row["split"]][row["id"]] if "split" in row else ALIGNED["test"][row["id"]]
        n_chunks = len(entry["bounds"]) - 1
        pieces = [cache["%s|%d" % (row["id"], k)] for k in range(n_chunks)]
        summaries.append(summ_utils.join_chunk_summaries(pieces))
    return summaries

In [ ]:
# Generate (and cache) every chunk summary first: test, then validation. A time cap stops cleanly and can be resumed.
if not STOPPED_EARLY and RUN_TEST and len(TEST_ROWS) > 0:
    if summarize_documents(TEST_ROWS, "test") is None:
        STOPPED_EARLY = True
if not STOPPED_EARLY and RUN_VAL and len(VAL_ROWS) > 0:
    if summarize_documents(VAL_ROWS, "validation") is None:
        STOPPED_EARLY = True
print("generation finished" if not STOPPED_EARLY else "generation NOT finished (time cap)")

## 9. Validation (held-out fold)

In [ ]:
VAL_RESULT = {}


def rouge_l_capped(pred, ref):
    p = " ".join(pred.split()[:ROUGE_L_MAX_WORDS])
    r = " ".join(ref.split()[:ROUGE_L_MAX_WORDS])
    return summ_utils.rougeL_f1(p, r)


if STOPPED_EARLY:
    print("Time cap reached before generation finished, skipping validation. Run again to resume.")
elif RUN_VAL and len(VAL_ROWS) > 0:
    val_summaries = summarize_documents(VAL_ROWS, "validation")
    r1 = []
    rl = []
    r1_lead = []
    rep = []
    ratio = []
    for row, pred in zip(VAL_ROWS, val_summaries):
        ref = row["summary"]
        r1.append(summ_utils.rouge1_f1(pred, ref))
        rl.append(rouge_l_capped(pred, ref))
        n_pred = len(pred.split())
        lead = " ".join(row["document"].split()[:max(n_pred, 1)])
        r1_lead.append(summ_utils.rouge1_f1(lead, ref))
        rep.append(summ_utils.repetition_rate(pred))
        ratio.append(n_pred / max(len(ref.split()), 1))

    VAL_RESULT = {
        "docs": len(VAL_ROWS),
        "rouge1_f1": float(np.mean(r1)),
        "rougeL_f1_capped": float(np.mean(rl)),
        "lead_same_length_rouge1_f1": float(np.mean(r1_lead)),
        "repetition_rate_3gram": float(np.mean(rep)),
        "pred_to_ref_length_ratio": float(np.mean(ratio)),
        "pred_words_mean": float(np.mean([len(p.split()) for p in val_summaries])),
        "ref_words_mean": float(np.mean([len(r["summary"].split()) for r in VAL_ROWS])),
    }
    print("\nvalidation (my ROUGE implementation, word level; not the official scorer):")
    for key in VAL_RESULT:
        print("  %-28s %.4f" % (key, VAL_RESULT[key]))

    if USE_BERTSCORE:
        try:
            from bert_score import score as bert_score_fn
            P, R, F1 = bert_score_fn(val_summaries, [r["summary"] for r in VAL_ROWS], lang="en", verbose=False)
            VAL_RESULT["bertscore_f1"] = float(F1.mean())
            print("  bertscore_f1                 %.4f" % VAL_RESULT["bertscore_f1"])
        except Exception as e:
            print("BERTScore skipped:", repr(e))

    with open(os.path.join(OUT_DIR, "val_predictions.json"), "w", encoding="utf-8") as f:
        json.dump([{"id": r["id"], "prediction": p, "reference": r["summary"]} for r, p in zip(VAL_ROWS, val_summaries)],
                  f, ensure_ascii=False, indent=1)
    print("saved val_predictions.json")
else:
    print("Validation skipped.")

## 10. Test predictions and submission

In [ ]:
SUBMISSION_PATH = os.path.join(OUT_DIR, "prediction.json")

if STOPPED_EARLY:
    print("Time cap reached before generation finished, skipping test predictions. Run again to resume.")
elif RUN_TEST and len(TEST_ROWS) > 0:
    test_summaries = summarize_documents(TEST_ROWS, "test")
    summaries = {}
    for row, text in zip(TEST_ROWS, test_summaries):
        summaries[row["id"]] = text
    sub = summ_utils.write_submission(TEST_ROWS, summaries, SUBMISSION_PATH)

    empty = sum(1 for s in sub if s["summary"].strip() == "")
    n_fallback = sum(1 for row in TEST_ROWS if summaries[row["id"]].strip() == "")
    print("summaries:", len(sub), "| empty generations replaced by the document start:", n_fallback)
    print("words per summary: mean %.0f | min %d | max %d" % (
        np.mean([len(s["summary"].split()) for s in sub]),
        min(len(s["summary"].split()) for s in sub), max(len(s["summary"].split()) for s in sub)))

    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(SUBMISSION_PATH, arcname="prediction.json")
    print("wrote", zip_path)
else:
    print("Test predictions skipped (no test docs yet, or RUN_TEST is off).")

## 11. Run summary (for the report)

In [ ]:
summary = {
    "model": "Knowledge distillation: student Phase-1 Model 1 + LoRA, teachers " + " + ".join(TEACHERS),
    "kd": {"teachers": TEACHERS, "alpha_ce": KD_ALPHA, "temperature": KD_TEMP, "train_budget_hours": TRAIN_BUDGET_HOURS,
           "budget_reached": BUDGET_REACHED},
    "strategy": "chunk-level summarization (sentence-aligned chunks), pieces joined in order",
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_layers": len(LORA_TARGETS)},
    "params_total": int(N_BASE),
    "params_trainable": int(N_LORA),
    "train_examples": len(PAIRS),
    "epochs": EPOCHS,
    "train_minutes": TRAIN_SECONDS / 60.0,
    "final_train_loss": LAST_LOSS,
    "validation": VAL_RESULT,
    "generation_tokens_per_second": GEN_STATS["tokens"] / max(GEN_STATS["seconds"], 1e-6),
    "peak_gpu_memory_gb": (torch.cuda.max_memory_allocated() / 1e9) if DEVICE.type == "cuda" else None,
    "stopped_early": STOPPED_EARLY,
    "chunk_doc_len": CHUNK_DOC_LEN,
    "chunk_sum_budget": CHUNK_SUM_BUDGET,
    "len_mult": LEN_MULT,
    "smoke_test": SMOKE_TEST,
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))

print("\nfiles in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  ", name, round(os.path.getsize(full) / 1e6, 2), "MB")